# Demo: Decisiones Accionables con Clustering + Transformer

Este notebook demuestra cómo combinar dos capas del sistema analítico:

- `Clustering AE/GMM`: aporta tipologías latentes de comportamiento y engagement.
- `Transformer`: aporta probabilidad predictiva de resultado académico o abandono.

El objetivo no es solo predecir, sino traducir ambas señales en **acciones priorizadas** para tutorización, seguimiento preventivo e intervenciones pedagógicas.

## Enfoque

La lógica del demo es la siguiente:

1. Localizar automáticamente la segmentación más reciente por ventana y split.
2. Localizar la predicción del transformer asociada al mismo split.
3. Reconciliar claves de estudiante para unir ambos mundos.
4. Derivar un `decision_score` y una `recommended_action` combinando:
   - riesgo predictivo del transformer,
   - pertenencia a cluster,
   - confianza/entropía de la segmentación,
   - metadatos básicos del estudiante.
5. Generar listados de intervención priorizada.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')

ROOT = Path('/workspace/TFM_education_ai_analytics')
DATA = ROOT / 'data'
PROCESSED = DATA / '2_processed'
SEGMENTED = DATA / '5_students_segmented'
PREDS = DATA / '7_model_predictions'

DEFAULT_SPLIT = 'validation'
DEFAULT_PRED_SCOPE = 'ternary_classification'

print({'root_exists': ROOT.exists(), 'processed_exists': PROCESSED.exists(), 'segmented_exists': SEGMENTED.exists(), 'preds_exists': PREDS.exists()})

In [ ]:
def discover_segmented_files(split: str = DEFAULT_SPLIT) -> pd.DataFrame:
    root = SEGMENTED / split
    if not root.exists():
        return pd.DataFrame(columns=['split', 'week', 'path'])

    rows = []
    for path in sorted(root.glob('students_segmented_uptoW*.csv')):
        m = re.search(r'uptoW(\d+)', path.name)
        if not m:
            continue
        rows.append({'split': split, 'week': int(m.group(1)), 'path': path})
    return pd.DataFrame(rows).sort_values('week') if rows else pd.DataFrame(columns=['split', 'week', 'path'])


def discover_prediction_files(scope: str = DEFAULT_PRED_SCOPE, split: str = DEFAULT_SPLIT) -> pd.DataFrame:
    root = PREDS / scope / split
    if not root.exists():
        return pd.DataFrame(columns=['scope', 'split', 'week', 'path'])

    rows = []
    for path in sorted(root.glob('transformer_preds_uptW*.csv')):
        m = re.search(r'uptW(\d+)', path.name)
        if not m:
            continue
        rows.append({'scope': scope, 'split': split, 'week': int(m.group(1)), 'path': path})
    return pd.DataFrame(rows).sort_values('week') if rows else pd.DataFrame(columns=['scope', 'split', 'week', 'path'])


seg_files = discover_segmented_files()
pred_files = discover_prediction_files()

display(seg_files.tail(10))
display(pred_files.tail(10))

## Selección de artefactos

Si existen varios experimentos, el demo intentará alinear segmentación y predicción por `split` y `week`. Cuando no haya coincidencia exacta, se detendrá con un mensaje explícito para evitar análisis engañosos.

In [ ]:
SELECTED_WEEK = None  # usa None para tomar la última intersección disponible
SPLIT = DEFAULT_SPLIT
PRED_SCOPE = DEFAULT_PRED_SCOPE

seg_files = discover_segmented_files(SPLIT)
pred_files = discover_prediction_files(PRED_SCOPE, SPLIT)

common_weeks = sorted(set(seg_files['week']).intersection(set(pred_files['week'])))
if not common_weeks:
    raise FileNotFoundError(
        'No hay una intersección entre segmentaciones y predicciones transformer para el split seleccionado. '
        'Ejecuta clustering + evaluate_transformer o cambia el scope/split.'
    )

week = common_weeks[-1] if SELECTED_WEEK is None else int(SELECTED_WEEK)
if week not in common_weeks:
    raise ValueError(f'La semana {week} no está disponible en ambos artefactos. Disponibles: {common_weeks}')

seg_path = seg_files.loc[seg_files['week'] == week, 'path'].iloc[-1]
pred_path = pred_files.loc[pred_files['week'] == week, 'path'].iloc[-1]

print({'split': SPLIT, 'scope': PRED_SCOPE, 'week': week, 'seg_path': str(seg_path), 'pred_path': str(pred_path)})

In [ ]:
def build_unique_id(df: pd.DataFrame) -> pd.Series:
    required = {'id_student', 'code_module', 'code_presentation'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'No puedo construir unique_id; faltan columnas: {sorted(missing)}')
    return (
        df['id_student'].astype(str)
        + '_' + df['code_module'].astype(str)
        + '_' + df['code_presentation'].astype(str)
    )


def load_student_metadata(split: str = SPLIT) -> pd.DataFrame:
    path = PROCESSED / split / 'students.csv'
    if not path.exists():
        raise FileNotFoundError(f'No existe {path}')

    stu = pd.read_csv(path)
    stu['unique_id'] = build_unique_id(stu)
    stu['id_student'] = stu['id_student'].astype(str)
    return stu


students_meta = load_student_metadata(SPLIT)
students_meta[['unique_id', 'id_student', 'code_module', 'code_presentation', 'final_result']].head()

In [ ]:
seg = pd.read_csv(seg_path)
pred = pd.read_csv(pred_path)

if 'unique_id' not in seg.columns:
    if 'Unnamed: 0' in seg.columns:
        seg = seg.rename(columns={'Unnamed: 0': 'unique_id'})
    else:
        raise ValueError('La segmentación no contiene unique_id y no puedo reconciliarla de forma segura.')

seg['unique_id'] = seg['unique_id'].astype(str)
pred['id_student'] = pred['id_student'].astype(str)

id_cardinality = students_meta.groupby('id_student')['unique_id'].nunique().rename('unique_id_count').reset_index()
pred = pred.merge(id_cardinality, on='id_student', how='left')

# En el caso ideal cada id_student aparece una sola vez en el split. Si no, marcamos ambigüedad.
pred_enriched = pred.merge(
    students_meta[['id_student', 'unique_id', 'code_module', 'code_presentation', 'final_result', 'gender', 'age_band', 'highest_education', 'studied_credits']].drop_duplicates(),
    on='id_student',
    how='left'
)
pred_enriched['join_status'] = np.where(pred_enriched['unique_id_count'].fillna(0).eq(1), 'ok', 'ambiguous_or_missing')

decision_df = pred_enriched.merge(seg, on='unique_id', how='left', suffixes=('_pred', '_cluster'))

print({'pred_rows': len(pred), 'seg_rows': len(seg), 'merged_rows': len(decision_df), 'safe_join_ratio': float((decision_df['join_status'] == 'ok').mean()) if len(decision_df) else np.nan})
decision_df.head()

## Política de decisión

La idea es transformar outputs analíticos en decisiones operativas. En esta demo:

- El **transformer** aporta el riesgo inmediato de abandono/fracaso.
- El **cluster** aporta el perfil estructural del estudiante (por ejemplo, inactividad crónica, exploración metódica, alto rendimiento, etc.).
- La combinación de ambos permite diferenciar entre intervención urgente, seguimiento preventivo, acompañamiento ligero o enriquecimiento académico.

In [ ]:
def infer_risk_probability(df: pd.DataFrame) -> pd.Series:
    candidate_cols = [
        'prob_fail_withdrawn',
        'prob_withdrawn',
        'prob_fail',
        'prob_fail_withdraw',
    ]

    for col in candidate_cols:
        if col in df.columns:
            return df[col].astype(float)

    prob_cols = [c for c in df.columns if c.startswith('prob_')]
    if 'pred_class_name' in df.columns and prob_cols:
        risk_like = [c for c in prob_cols if any(token in c for token in ['withdraw', 'fail', 'risk'])]
        if risk_like:
            return df[risk_like].max(axis=1).astype(float)

    raise ValueError('No he podido inferir una probabilidad de riesgo desde las columnas de predicción.')


def cluster_risk_boost(name: str) -> float:
    if not isinstance(name, str):
        return 0.0
    name = name.lower()
    if 'critical' in name or 'inactivo' in name or 'riesgo' in name:
        return 0.20
    if 'fatigue' in name or 'explorer' in name or 'standard' in name:
        return 0.08
    if 'good' in name or 'high' in name or 'performer' in name:
        return -0.08
    return 0.0


def recommend_action(row: pd.Series) -> str:
    score = row['decision_score']
    if score >= 0.80:
        return 'Tutor outreach inmediato + revisión de actividad + contacto humano'
    if score >= 0.60:
        return 'Intervención preventiva esta semana + recordatorio evaluaciones'
    if score >= 0.40:
        return 'Seguimiento ligero + nudges personalizados'
    return 'Mantener seguimiento estándar / posible itinerario de enriquecimiento'


decision_df = decision_df.copy()
decision_df['risk_probability'] = infer_risk_probability(decision_df)
decision_df['cluster_boost'] = decision_df.get('cluster_name', pd.Series(index=decision_df.index, dtype=object)).map(cluster_risk_boost).fillna(0.0)
decision_df['cluster_uncertainty_penalty'] = decision_df.get('entropy_norm', pd.Series(0.0, index=decision_df.index)).fillna(0.0) * 0.05
decision_df['decision_score'] = (decision_df['risk_probability'] + decision_df['cluster_boost'] - decision_df['cluster_uncertainty_penalty']).clip(0.0, 1.0)
decision_df['recommended_action'] = decision_df.apply(recommend_action, axis=1)
decision_df['priority_band'] = pd.cut(
    decision_df['decision_score'],
    bins=[-0.01, 0.40, 0.60, 0.80, 1.0],
    labels=['monitor', 'light_intervention', 'preventive', 'urgent']
)

decision_df[['unique_id', 'id_student', 'cluster_name', 'pred_class_name', 'risk_probability', 'decision_score', 'priority_band', 'recommended_action']].head(10)

In [ ]:
summary = (
    decision_df.groupby(['priority_band', 'recommended_action'], dropna=False)
    .size()
    .rename('n_students')
    .reset_index()
    .sort_values(['priority_band', 'n_students'], ascending=[True, False])
)

display(summary)

top_cases = decision_df.sort_values('decision_score', ascending=False)[[
    'unique_id', 'id_student', 'code_module', 'code_presentation', 'final_result',
    'cluster_name', 'pred_class_name', 'risk_probability', 'decision_score', 'recommended_action', 'join_status'
]].head(25)

display(top_cases)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

priority_counts = decision_df['priority_band'].value_counts(dropna=False).reindex(['urgent', 'preventive', 'light_intervention', 'monitor'])
priority_counts.plot(kind='bar', ax=axes[0], color=['#b22222', '#ff8c00', '#4682b4', '#2e8b57'])
axes[0].set_title('Distribución de prioridades de intervención')
axes[0].set_xlabel('Banda de prioridad')
axes[0].set_ylabel('N estudiantes')

if 'cluster_name' in decision_df.columns and decision_df['cluster_name'].notna().any():
    heat = (
        decision_df.pivot_table(
            index='cluster_name',
            columns='priority_band',
            values='decision_score',
            aggfunc='mean'
        )
        .reindex(columns=['urgent', 'preventive', 'light_intervention', 'monitor'])
    )
    sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Score medio por cluster y prioridad')
else:
    axes[1].text(0.5, 0.5, 'Sin información de cluster disponible', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## Interpretación sugerida

Este demo propone una lógica de decisión en dos niveles:

- **Riesgo coyuntural**: lo detecta el transformer mediante probabilidad predictiva.
- **Perfil estructural**: lo aporta el clustering, que añade contexto comportamental al riesgo.

Así, dos estudiantes con la misma probabilidad de riesgo pueden requerir acciones distintas si uno pertenece a un cluster de inactividad persistente y otro a un cluster de alto compromiso con señales recientes de deterioro. La utilidad real del sistema aparece cuando ambas señales se combinan para priorizar recursos tutoriales limitados.

## Próximos pasos

Ideas para endurecer este notebook y convertirlo en una pieza de demo más seria:

- Exportar el `unique_id` también desde las predicciones transformer para eliminar ambigüedades de unión.
- Incorporar coste de intervención y capacidad limitada del equipo docente.
- Medir precisión de las decisiones recomendadas frente a resultados observados.
- Añadir reglas distintas por escenario (`ternary`, `binary_paper`, `binary_pass_vs_withdraw`).